# Advanced Machine Learning (21CSE425T) — Unit 5: Neural Networks

Standalone Colab notebook for **T13–T15**: optimizer comparison with Dropout/BatchNorm, back-propagation from scratch, and deep-NN Face Recognition. Run top-to-bottom; the setup cell installs all dependencies.

---
# Unit 5 — Neural Networks

Installs torch/torchvision/matplotlib/scikit-learn, then covers **optimizer comparison with Dropout &
BatchNorm** (T13), **back-propagation implemented from scratch** (T14), and **deep-NN face recognition**
(T15).

### Unit 5 — Setup & installs

In [ ]:
# torch & torchvision ship with Colab; install only if missing.
!pip -q install torchvision matplotlib scikit-learn 2>/dev/null
import torch, torchvision, matplotlib, sklearn
print("PyTorch    :", torch.__version__)
print("torchvision:", torchvision.__version__)
print("Device     :", DEVICE)

# Device handle reused by the PyTorch tasks in this notebook.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

## T13 — Optimizer comparison (SGD vs Adam vs RMSProp vs Adagrad) + Dropout & BatchNorm

The **same MLP** (with Dropout and Batch Normalisation) is trained on `sklearn` **digits** (8×8) under four
optimizers. We plot training loss and test accuracy so the convergence behaviour of each optimizer is
directly comparable.

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(0); np.random.seed(0)

digits = load_digits()
Xtr, Xte, ytr, yte = train_test_split(digits.data, digits.target,
                                      test_size=0.25, random_state=0, stratify=digits.target)
sc = StandardScaler().fit(Xtr)
Xtr_t = torch.tensor(sc.transform(Xtr), dtype=torch.float32, device=DEVICE)
Xte_t = torch.tensor(sc.transform(Xte), dtype=torch.float32, device=DEVICE)
ytr_t = torch.tensor(ytr, dtype=torch.long, device=DEVICE)
yte_t = torch.tensor(yte, dtype=torch.long, device=DEVICE)

class MLP(nn.Module):
    """MLP demonstrating Dropout + BatchNorm regularisation."""
    def __init__(self, in_dim=64, hidden=128, n_classes=10, p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(p),
            nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(p),
            nn.Linear(hidden, n_classes),
        )
    def forward(self, x):
        return self.net(x)

def make_optimizer(name, params, lr=0.01):
    params = list(params)   # materialize: a generator is exhausted by the first optimizer built below
    return {
        "SGD":     torch.optim.SGD(params, lr=lr, momentum=0.9),
        "Adam":    torch.optim.Adam(params, lr=lr),
        "RMSProp": torch.optim.RMSprop(params, lr=lr),
        "Adagrad": torch.optim.Adagrad(params, lr=lr),
    }[name]

def train_with(opt_name, epochs=120):
    torch.manual_seed(0)
    model = MLP().to(DEVICE)
    opt = make_optimizer(opt_name, model.parameters())
    loss_fn = nn.CrossEntropyLoss()
    losses, accs = [], []
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss = loss_fn(model(Xtr_t), ytr_t)
        loss.backward(); opt.step()
        losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            acc = (model(Xte_t).argmax(1) == yte_t).float().mean().item()
        accs.append(acc)
    return losses, accs

results = {name: train_with(name) for name in ["SGD", "Adam", "RMSProp", "Adagrad"]}
for name, (losses, accs) in results.items():
    print(f"{name:8s} | final loss {losses[-1]:.4f} | final test acc {accs[-1]:.3f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for name, (losses, accs) in results.items():
    ax1.plot(losses, label=name)
    ax2.plot(accs,  label=name)
ax1.set_title("T13 — Training loss"); ax1.set_xlabel("epoch"); ax1.set_ylabel("loss")
ax2.set_title("T13 — Test accuracy"); ax2.set_xlabel("epoch"); ax2.set_ylabel("accuracy")
for a in (ax1, ax2): a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## T14 — Back-propagation from scratch (NumPy MLP)

A multi-layer perceptron with **no autograd** — forward pass, manual gradient derivation through the chain
rule, and gradient-descent updates — trained on the non-linearly-separable **two-moons** dataset. The
learned decision boundary confirms the hand-coded back-prop works.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

np.random.seed(42)
X, y = make_moons(n_samples=600, noise=0.2, random_state=42)
y = y.reshape(-1, 1)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)

def sigmoid(z):  return 1.0 / (1.0 + np.exp(-z))
def d_sigmoid(a): return a * (1.0 - a)          # derivative in terms of the activation
def relu(z):     return np.maximum(0, z)
def d_relu(z):   return (z > 0).astype(float)

class ScratchMLP:
    """2-layer MLP (ReLU hidden, sigmoid output) with hand-coded back-propagation."""
    def __init__(self, n_in, n_hidden, n_out=1, lr=0.1):
        self.lr = lr
        # He-ish initialisation
        self.W1 = np.random.randn(n_in, n_hidden) * np.sqrt(2.0 / n_in)
        self.b1 = np.zeros((1, n_hidden))
        self.W2 = np.random.randn(n_hidden, n_out) * np.sqrt(2.0 / n_hidden)
        self.b2 = np.zeros((1, n_out))

    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1          # pre-activation hidden
        self.a1 = relu(self.z1)                  # hidden activation
        self.z2 = self.a1 @ self.W2 + self.b2    # pre-activation output
        self.a2 = sigmoid(self.z2)               # predicted probability
        return self.a2

    def backward(self, X, y):
        m = X.shape[0]
        # ---- output layer: dL/dz2 for binary cross-entropy + sigmoid = (a2 - y) ----
        dz2 = (self.a2 - y) / m
        dW2 = self.a1.T @ dz2
        db2 = dz2.sum(axis=0, keepdims=True)
        # ---- hidden layer: propagate error back through W2 and ReLU ----
        da1 = dz2 @ self.W2.T
        dz1 = da1 * d_relu(self.z1)
        dW1 = X.T @ dz1
        db1 = dz1.sum(axis=0, keepdims=True)
        # ---- gradient-descent update ----
        self.W2 -= self.lr * dW2; self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1; self.b1 -= self.lr * db1

    def loss(self, y):
        eps = 1e-9
        return float(-np.mean(y*np.log(self.a2+eps) + (1-y)*np.log(1-self.a2+eps)))

net = ScratchMLP(n_in=2, n_hidden=16, lr=0.5)
history = []
for epoch in range(2000):
    net.forward(Xtr); net.backward(Xtr, ytr)
    if epoch % 100 == 0:
        history.append(net.loss(ytr))

acc = float(((net.forward(Xte) > 0.5).astype(int) == yte).mean())
print(f"Test accuracy (scratch back-prop MLP): {acc:.3f}")

In [ ]:
# ---- Visualise loss curve and the learned decision boundary ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(range(0, 2000, 100), history, marker="o", ms=3)
ax1.set_title("T14 — Training loss (from-scratch back-prop)")
ax1.set_xlabel("epoch"); ax1.set_ylabel("BCE loss"); ax1.grid(alpha=0.3)

xx, yy = np.meshgrid(np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 300),
                     np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = (net.forward(grid) > 0.5).reshape(xx.shape)
ax2.contourf(xx, yy, zz, alpha=0.3, cmap="coolwarm")
ax2.scatter(X[:,0], X[:,1], c=y.ravel(), cmap="coolwarm", edgecolors="k", s=18)
ax2.set_title("T14 — Learned decision boundary (two moons)")
plt.tight_layout(); plt.show()

## T15 — Face recognition with a deep neural network (CNN)

A compact **convolutional neural network** in PyTorch recognises identities in the **Olivetti faces**
dataset — convolutional feature extraction + a classifier head trained end-to-end, contrasting with the
PCA+ensemble pipeline of T9.

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split

torch.manual_seed(0); np.random.seed(0)

faces = fetch_olivetti_faces(shuffle=True, random_state=0)
X = faces.images[:, None, :, :].astype("float32")   # (N,1,64,64)
y = faces.target.astype("int64")
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

Xtr_t = torch.tensor(Xtr, device=DEVICE); ytr_t = torch.tensor(ytr, device=DEVICE)
Xte_t = torch.tensor(Xte, device=DEVICE); yte_t = torch.tensor(yte, device=DEVICE)

class FaceCNN(nn.Module):
    def __init__(self, n_classes=40):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),  # 64->32
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2), # 32->16
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2), # 16->8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.4),
            nn.Linear(64*8*8, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, n_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model = FaceCNN(n_classes=int(y.max())+1).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

bs = 32
for epoch in range(1, 41):
    model.train()
    perm = torch.randperm(Xtr_t.size(0))
    for i in range(0, Xtr_t.size(0), bs):
        idx = perm[i:i+bs]
        opt.zero_grad()
        loss = loss_fn(model(Xtr_t[idx]), ytr_t[idx])
        loss.backward(); opt.step()
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            acc = (model(Xte_t).argmax(1) == yte_t).float().mean().item()
        print(f"epoch {epoch:2d} | loss {loss.item():.4f} | test acc {acc:.3f}")

In [ ]:
# ---- Show CNN predictions ----
model.eval()
with torch.no_grad():
    preds = model(Xte_t).argmax(1).cpu().numpy()

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for ax, img, p, t in zip(axes.ravel(), Xte, preds, yte):
    ax.imshow(img[0], cmap="gray")
    ok = (p == t)
    ax.set_title(f"pred {p}\n{'OK' if ok else 'true '+str(t)}",
                 color=("green" if ok else "red"), fontsize=9)
    ax.axis("off")
fig.suptitle("T15 — Deep CNN face recognition (Olivetti)")
plt.tight_layout(); plt.show()

---
### Notebook complete — T1–T15 across all 5 units

**Tips for Colab**
- `Runtime → Change runtime type → GPU` speeds up Units 1 and 5.
- Each unit's **install cell is independent**, so you can run units in any order.
- Datasets used are either built-in or auto-downloaded (Olivetti faces, Karate Club); no manual uploads needed.